# Crawler BRACIS (Essencial)

Coleta trabalhos das edicoes de 2023, 2024 e 2025 do BRACIS no SOL/SBC e consolida em `DataFrame`.

In [26]:
import re
import time
from dataclasses import dataclass
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://sol.sbc.org.br/index.php"
BRACIS_SLUG = "bracis"
ANOS_ALVO = {2023, 2024, 2025}
REQUEST_TIMEOUT = 30
SLEEP_SECONDS = 0.2
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)


In [27]:
@dataclass(frozen=True)
class Edicao:
    ano: int
    titulo: str
    link: str


def baixar_soup(url: str, session: requests.Session) -> BeautifulSoup:
    resposta = session.get(url, timeout=REQUEST_TIMEOUT)
    resposta.raise_for_status()
    return BeautifulSoup(resposta.text, "html.parser")


def extrair_ano(texto: str):
    if not texto:
        return None
    match = re.search(r"\b(20\d{2})\b", texto)
    return int(match.group(1)) if match else None


def listar_edicoes_bracis(session: requests.Session, anos_alvo):
    archive_url = f"{BASE_URL}/{BRACIS_SLUG}/issue/archive"
    soup = baixar_soup(archive_url, session)
    edicoes = []

    for bloco in soup.select("div.obj_issue_summary"):
        tag_titulo = bloco.select_one("a.title")
        if not tag_titulo:
            continue

        titulo_edicao = tag_titulo.get_text(" ", strip=True)
        ano_edicao = extrair_ano(titulo_edicao)

        if ano_edicao not in anos_alvo:
            continue

        link_edicao = urljoin(archive_url, tag_titulo.get("href", "").strip())
        edicoes.append(Edicao(ano=ano_edicao, titulo=titulo_edicao, link=link_edicao))

    edicoes.sort(key=lambda item: (item.ano, item.titulo), reverse=True)
    return edicoes


def listar_artigos_edicao(url_edicao: str, session: requests.Session):
    soup = baixar_soup(url_edicao, session)
    artigos = []
    links_vistos = set()

    for bloco in soup.select("div.obj_article_summary"):
        tag_artigo = bloco.select_one("div.title a")
        if not tag_artigo:
            continue

        link_artigo = urljoin(url_edicao, tag_artigo.get("href", "").strip())
        if not link_artigo or link_artigo in links_vistos:
            continue

        links_vistos.add(link_artigo)
        artigos.append(
            {
                "titulo_edicao": tag_artigo.get_text(" ", strip=True),
                "link_artigo": link_artigo,
            }
        )

    return artigos


def extrair_titulo_resumo(url_artigo: str, titulo_fallback: str, session: requests.Session):
    soup = baixar_soup(url_artigo, session)

    tag_titulo = soup.select_one("h1.page_title")
    titulo = tag_titulo.get_text(" ", strip=True) if tag_titulo else titulo_fallback

    resumo = ""
    tag_resumo = soup.select_one("div.item.abstract")
    if tag_resumo:
        tag_label = tag_resumo.select_one("h3.label")
        if tag_label:
            tag_label.decompose()
        resumo = tag_resumo.get_text(" ", strip=True)

    return titulo, resumo


def coletar_trabalhos_bracis(anos_alvo=ANOS_ALVO, sleep_seconds=SLEEP_SECONDS):
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    registros = []
    edicoes = listar_edicoes_bracis(session, anos_alvo)

    print(f"Edicoes BRACIS encontradas para {sorted(anos_alvo)}: {len(edicoes)}")

    for edicao in edicoes:
        print(f"- {edicao.ano}: {edicao.titulo}")
        artigos = listar_artigos_edicao(edicao.link, session)
        print(f"  Artigos na edicao: {len(artigos)}")

        total_artigos = len(artigos)
        for indice, artigo in enumerate(artigos, start=1):
            time.sleep(sleep_seconds)
            try:
                titulo, resumo = extrair_titulo_resumo(
                    artigo["link_artigo"], artigo["titulo_edicao"], session
                )
            except requests.RequestException as erro:
                print(f"  [aviso] erro em {artigo['link_artigo']}: {erro}")
                continue

            registros.append(
                {
                    "Conferencia": "BRACIS",
                    "Ano": edicao.ano,
                    "Edicao": edicao.titulo,
                    "Titulo": titulo,
                    "Resumo": resumo,
                    "Link": artigo["link_artigo"],
                }
            )

            if indice % 25 == 0 or indice == total_artigos:
                print(f"  Progresso: {indice}/{total_artigos}")

    df = pd.DataFrame(
        registros,
        columns=["Conferencia", "Ano", "Edicao", "Titulo", "Resumo", "Link"],
    )

    if not df.empty:
        df = (
            df.drop_duplicates(subset=["Ano", "Titulo", "Link"])
            .sort_values(["Ano", "Titulo"], ascending=[False, True])
            .reset_index(drop=True)
        )

    return df


In [28]:
df_bracis = coletar_trabalhos_bracis(anos_alvo={2023, 2024, 2025}, sleep_seconds=0.2)
df_bracis.head(10)

Edicoes BRACIS encontradas para [2023, 2024, 2025]: 3
- 2025: 2025: Anais da XXXV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 148
  Progresso: 25/148
  Progresso: 50/148
  Progresso: 75/148
  Progresso: 100/148
  Progresso: 125/148
  Progresso: 148/148
- 2024: 2024: Anais da XXXIV Brazilian Conference on Intelligent Systems
  Artigos na edicao: 116
  Progresso: 25/116
  Progresso: 50/116
  Progresso: 75/116
  Progresso: 100/116
  Progresso: 116/116
- 2023: 2023: Anais da XII Brazilian Conference on Intelligent Systems
  Artigos na edicao: 90
  Progresso: 25/90
  Progresso: 50/90
  Progresso: 75/90
  Progresso: 90/90


,Conferencia,Ano,Edicao,Titulo,Resumo,Link
0,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Categorical Kalman Filter for Human Activity...,Human Activity Recognition (HAR) has gained in...,https://sol.sbc.org.br/index.php/bracis/articl...
1,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comparative Study of Machine Learning Models...,The bee flora refers to the group of plants fr...,https://sol.sbc.org.br/index.php/bracis/articl...
2,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Comprehensive Study of Fitness Landscapes in...,The search space in Automated Machine Learning...,https://sol.sbc.org.br/index.php/bracis/articl...
3,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Double Transfer Learning Approach for Improv...,Cancer is one of the most devastating health c...,https://sol.sbc.org.br/index.php/bracis/articl...
4,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Fully Automatic Approach for COVID-19 Diagno...,The pandemic caused by the COVID-19 virus has ...,https://sol.sbc.org.br/index.php/bracis/articl...
5,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Generative Domain Adaptation Scheme for Swif...,Deep learning models have demonstrated remarka...,https://sol.sbc.org.br/index.php/bracis/articl...
6,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Livestock Monitoring System and Machine Lear...,Timely disease diagnosis is essential for redu...,https://sol.sbc.org.br/index.php/bracis/articl...
7,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multi-Dimensional Comparative Study of Gener...,Synthetic data is increasingly important in pr...,https://sol.sbc.org.br/index.php/bracis/articl...
8,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A Multidimensional Analysis of Swarm Dynamics ...,Traditional assessments of metaheuristics typi...,https://sol.sbc.org.br/index.php/bracis/articl...
9,BRACIS,2025,2025: Anais da XXXV Brazilian Conference on In...,A NAS-Optimized Deep Learning Model for Custom...,Deep Learning (DL) methods offer competitive a...,https://sol.sbc.org.br/index.php/bracis/articl...


In [29]:
print(f"Total de registros: {len(df_bracis)}")
print("\nColunas:", list(df_bracis.columns))
print("\nTrabalhos por ano:")
print(df_bracis["Ano"].value_counts().sort_index(ascending=False))

Total de registros: 354

Colunas: ['Conferencia', 'Ano', 'Edicao', 'Titulo', 'Resumo', 'Link']

Trabalhos por ano:
Ano
2025    148
2024    116
2023     90
Name: count, dtype: int64


In [30]:
arquivo_saida = "bracis_trabalhos_2023_2025.csv"
df_bracis.to_csv(arquivo_saida, index=False)
print(f"CSV salvo em: {arquivo_saida}")

CSV salvo em: bracis_trabalhos_2023_2025.csv


## Parte 1 - Bag of Words, TF-IDF e Modelagem de Topicos

Secao para a atividade:

- (a) representacoes BoW e TF-IDF;
- (b) modelagem de topicos com NMF (e LDA opcional);
- (c) interpretacao dos topicos.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

pd.set_option("display.max_colwidth", 220)

In [ ]:
if "df_bracis" not in globals() or df_bracis is None or len(df_bracis) == 0:
    df_bracis = pd.read_csv("bracis_trabalhos_2023_2025.csv")

df_nlp = df_bracis.copy()
for coluna in ["Titulo", "Resumo"]:
    df_nlp[coluna] = (
        df_nlp[coluna]
        .fillna("")
        .astype(str)
        .str.replace(r"\\s+", " ", regex=True)
        .str.strip()
    )

df_nlp["texto"] = (df_nlp["Titulo"] + ". " + df_nlp["Resumo"]).str.strip()
df_nlp = df_nlp[df_nlp["texto"].str.len() > 0].reset_index(drop=True)

print(f"Documentos no corpus: {len(df_nlp)}")
df_nlp[["Ano", "Titulo", "Resumo"]].head(3)

In [ ]:
# (a) Bag of Words
bow_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
)

X_bow = bow_vectorizer.fit_transform(df_nlp["texto"])
vocab_bow = bow_vectorizer.get_feature_names_out()

freq_bow = np.asarray(X_bow.sum(axis=0)).ravel()
idx_top_bow = freq_bow.argsort()[::-1][:25]
bow_top_terms = pd.DataFrame(
    {
        "termo": vocab_bow[idx_top_bow],
        "frequencia": freq_bow[idx_top_bow],
    }
)

print("Matriz BoW:", X_bow.shape)
print("Vocabulario BoW:", len(vocab_bow))
bow_top_terms

In [ ]:
# (a) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X_tfidf = tfidf_vectorizer.fit_transform(df_nlp["texto"])
vocab_tfidf = tfidf_vectorizer.get_feature_names_out()

mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
idx_top_tfidf = mean_tfidf.argsort()[::-1][:25]
tfidf_top_terms = pd.DataFrame(
    {
        "termo": vocab_tfidf[idx_top_tfidf],
        "peso_medio_tfidf": mean_tfidf[idx_top_tfidf],
    }
)

print("Matriz TF-IDF:", X_tfidf.shape)
print("Vocabulario TF-IDF:", len(vocab_tfidf))
tfidf_top_terms

In [ ]:
def top_words_by_topic(components, feature_names, n_top_words=12):
    linhas = []
    for topic_id, topic_weights in enumerate(components):
        top_idx = topic_weights.argsort()[::-1][:n_top_words]
        palavras = [feature_names[i] for i in top_idx]
        linhas.append(
            {
                "topico": topic_id,
                "palavras_chave": ", ".join(palavras),
            }
        )
    return pd.DataFrame(linhas)


In [ ]:
# (b) NMF sobre TF-IDF (preferencial)
N_TOPICS = 8
N_TOP_WORDS = 12

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvda",
    max_iter=600,
)

W_nmf = nmf_model.fit_transform(X_tfidf)
H_nmf = nmf_model.components_

nmf_topics_df = top_words_by_topic(H_nmf, vocab_tfidf, n_top_words=N_TOP_WORDS)
df_nlp["topico_nmf"] = W_nmf.argmax(axis=1)
df_nlp["score_topico_nmf"] = W_nmf.max(axis=1)

print("Erro de reconstrucao (NMF):", round(nmf_model.reconstruction_err_, 4))
nmf_topics_df

In [ ]:
# (b) LDA opcional sobre BoW
lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch",
    max_iter=25,
)

W_lda = lda_model.fit_transform(X_bow)
H_lda = lda_model.components_

lda_topics_df = top_words_by_topic(H_lda, vocab_bow, n_top_words=N_TOP_WORDS)
df_nlp["topico_lda"] = W_lda.argmax(axis=1)

print("Perplexidade (LDA):", round(lda_model.perplexity(X_bow), 2))
lda_topics_df

In [ ]:
# (c) Interpretacao dos topicos (NMF)
def interpretar_topicos_nmf(df_docs, topicos_df, top_n_titulos=3):
    linhas = []
    for _, linha in topicos_df.iterrows():
        t = int(linha["topico"])
        sub = df_docs[df_docs["topico_nmf"] == t].copy()

        qtd_docs = len(sub)
        por_ano = sub["Ano"].value_counts().sort_index().to_dict()
        titulos_rep = (
            sub.sort_values("score_topico_nmf", ascending=False)["Titulo"]
            .head(top_n_titulos)
            .tolist()
        )

        termos = linha["palavras_chave"].split(", ")[:4]
        interpretacao = (
            f"Topico {t} foca em {', '.join(termos)}. "
            f"Documentos: {qtd_docs}. Distribuicao por ano: {por_ano}."
        )

        linhas.append(
            {
                "topico": t,
                "palavras_chave": linha["palavras_chave"],
                "qtd_documentos": qtd_docs,
                "distribuicao_anos": por_ano,
                "titulos_representativos": " | ".join(titulos_rep),
                "interpretacao_inicial": interpretacao,
            }
        )

    return pd.DataFrame(linhas).sort_values("qtd_documentos", ascending=False)


interpretacao_nmf_df = interpretar_topicos_nmf(df_nlp, nmf_topics_df)
interpretacao_nmf_df

### Texto para relatorio (sugestao)

Use a tabela `interpretacao_nmf_df` para descrever cada topico no relatorio.

Estrutura sugerida:

1. Palavras-chave do topico.
2. Quantidade de trabalhos e distribuicao por ano.
3. Titulos representativos.
4. Interpretacao semantica final (refinada por voce ou por LLM).